# Longitudinal infant gut microbiome: alpha and beta diversity trajectories

**Purpose.** Complements the cross-sectional (grouped-by-age) analyses by following the *same infants* over time.
It asks three questions:

1. Does Shannon diversity increase with age, and what does this look like per infant? (Section 2)
2. Are samples from the same infant more similar than samples from different infants, once age is accounted for? (Section 3)
3. How much of the Bray–Curtis variation is explained by age vs. infant identity? (Section 4)

**Inputs** (in `micro_odyssey_data/`): `shannon_vector.qza`, `metadata_numage.tsv`, `bracken-table.qza`, and the QIIME 2 PERMANOVA/PERMDISP `.qzv` files.
**Outputs:** figures are written to `figures/`.

> Run order matters: use *Kernel → Restart & Run All* before submitting so the cell numbering is clean and everything is reproducible.

## 0. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import biom
import qiime2 as q2
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy.spatial.distance import pdist, squareform

%matplotlib inline

data_dir = Path("micro_odyssey_data")
fig_dir = Path("figures")
fig_dir.mkdir(exist_ok=True)

# Shared plotting conventions (used by every figure below)
sns.set_theme(style="whitegrid", context="talk")
AGES = [0, 2, 6, 12, 24]                                   # sampling time points (months)
AGE_LABELS = ["Birth", "2 months", "6 months", "12 months", "24 months"]
AGE_PALETTE = sns.color_palette("mako", len(AGES))         # colour = age
COMP_COLORS = {"Same infant": "#4A9D9D", "Different infants": "#53649A"}  # colour = comparison type
DARK = "#17253A"

def save_fig(name):
    """Save the current figure to figures/ at publication resolution."""
    plt.savefig(fig_dir / name, dpi=300, bbox_inches="tight")

## 1. Load data (once)

In [5]:
# Alpha diversity (Shannon) per sample
shannon = q2.Artifact.load(data_dir / "shannon_vector.qza").view(pd.Series).rename("Shannon")

# Sample metadata
meta = pd.read_csv(data_dir / "metadata_numage.tsv", sep="\t", index_col=0)

# Taxonomic abundance table (Bracken): samples x taxa
biom_table = q2.Artifact.load(data_dir / "bracken-table.qza").view(biom.Table)
abundance = biom_table.to_dataframe(dense=True).T

print("Abundance table (samples x taxa):", abundance.shape)
print("Samples with metadata:", abundance.index.isin(meta.index).sum(), "/", len(abundance))

ValueError: micro_odyssey_data/shannon_vector.qza does not exist.

## 2. Alpha diversity across infant development

### 2.1 Cross-sectional view: Shannon diversity by age group
The violin plot groups samples by age but does not show which samples belong to the same infant,
so it cannot say whether individual infants follow different trajectories (see 2.3).

In [6]:
# One table used for all alpha-diversity analyses: Shannon + age + infant ID
alpha = shannon.to_frame().join(meta[["age_months", "host_id"]]).dropna()
alpha["Age"] = pd.Categorical(
    alpha["age_months"].astype(int).map(dict(zip(AGES, AGE_LABELS))),
    categories=AGE_LABELS, ordered=True,
)
print(f"{len(alpha)} samples from {alpha['host_id'].nunique()} infants")
display(alpha.head())

NameError: name 'shannon' is not defined

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.violinplot(
    data=alpha, x="Age", y="Shannon", order=AGE_LABELS,
    hue="Age", palette=AGE_PALETTE, legend=False,
    inner="box", cut=0, linewidth=1.2, ax=ax,
)
sns.stripplot(
    data=alpha, x="Age", y="Shannon", order=AGE_LABELS,
    color="black", size=3, alpha=0.35, jitter=0.18, ax=ax,
)

ax.set_xlabel("Infant age")
ax.set_ylabel("Shannon diversity")
ax.set_title("Gut microbiome alpha diversity across infant development", fontsize=16, weight="bold")
sns.despine()
plt.tight_layout()
save_fig("shannon_violin.png")
plt.show()

### 2.2 Mixed-effects model: Shannon ~ age, random intercept per infant
The random intercept accounts for repeated sampling of the same infant (samples are not independent).
`age_months` is treated as a continuous, linear effect; the time points are unevenly spaced (0, 2, 6, 12, 24),
so check that a linear trend is reasonable (e.g., compare against a model with `np.log1p(age_months)` or a random slope).

In [ ]:
lme = smf.mixedlm("Shannon ~ age_months", data=alpha, groups=alpha["host_id"]).fit(reml=True)
print(lme.summary())

### 2.3 Individual infant trajectories
Thin lines connect samples from the same infant; the dark line is the mean per age.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.5))

# 1. Individual infant trajectories (light)
for _, grp in alpha.groupby("host_id"):
    grp = grp.sort_values("age_months")
    ax.plot(grp["age_months"], grp["Shannon"], color="#7393B3", alpha=0.16, linewidth=1.2, zorder=1)

# 2. Individual samples, coloured by age
for age, colour in zip(AGES, AGE_PALETTE):
    sub = alpha[alpha["age_months"] == age]
    ax.scatter(sub["age_months"], sub["Shannon"], color=colour, alpha=0.55, s=28, edgecolors="none", zorder=2)

# 3. Mean per age
mean_shannon = alpha.groupby("age_months")["Shannon"].mean().reindex(AGES)
ax.plot(AGES, mean_shannon, color=DARK, linewidth=3.2, marker="o", markersize=8,
        markerfacecolor="white", markeredgewidth=2.5, label="Mean Shannon diversity", zorder=5)

ax.set_xticks(AGES)
ax.set_xticklabels(["Birth", "2", "6", "12", "24"])
ax.set_xlabel("Infant age (months)")
ax.set_ylabel("Shannon diversity")
ax.set_title("Individual gut microbiome diversity trajectories", fontsize=17, fontweight="bold", pad=18)
ax.legend(frameon=False, loc="upper left", fontsize=11)
ax.grid(axis="x", visible=False)
sns.despine()
plt.tight_layout()
save_fig("shannon_trajectories.png")
plt.show()

## 3. Beta diversity: same infant vs. different infants

### 3.1 Bray–Curtis distance matrix from the Bracken table
Counts are converted to relative abundances, then pairwise Bray–Curtis dissimilarities are computed.
(Sanity check worth doing once: this matrix should match the QIIME 2 Bray–Curtis distance matrix used for the PERMANOVA below.)

In [ ]:
relative_abundance = abundance.div(abundance.sum(axis=1), axis=0)

bray = pd.DataFrame(
    squareform(pdist(relative_abundance, metric="braycurtis")),
    index=abundance.index, columns=abundance.index,
)
print("Bray-Curtis distance matrix:", bray.shape)

### 3.2 Build one table of all sample pairs
Every pairwise comparison is computed **once** here and reused by all plots below. Columns:
infant and age of each sample, whether the pair is from the same infant, the age gap, and the distance.
Note that pairs are *not independent* (each sample appears in many pairs), so boxplots are descriptive;
formal tests should use permutation/mixed-model approaches (Section 5).

In [ ]:
# Keep samples that are in the distance matrix and have complete metadata
info = meta.loc[bray.index.intersection(meta.index), ["host_id", "age_months"]].dropna()
D = bray.loc[info.index, info.index].to_numpy()

# All unique sample pairs (upper triangle)
i, j = np.triu_indices(len(info), k=1)
host = info["host_id"].to_numpy()
age = info["age_months"].to_numpy().astype(int)

pairs_all = pd.DataFrame({
    "Sample1": info.index[i], "Sample2": info.index[j],
    "Same_infant": host[i] == host[j],
    "Age1": np.minimum(age[i], age[j]),     # younger sample of the pair
    "Age2": np.maximum(age[i], age[j]),     # older sample of the pair
    "Bray_Curtis": D[i, j],
})
pairs_all["Age_gap"] = pairs_all["Age2"] - pairs_all["Age1"]
pairs_all["Age_pair"] = pairs_all["Age1"].astype(str) + "–" + pairs_all["Age2"].astype(str)
pairs_all["Comparison"] = np.where(pairs_all["Same_infant"], "Same infant", "Different infants")

# Pairs at *different* ages: the fair like-for-like comparison of same vs different infants
pairs = pairs_all[pairs_all["Age_gap"] > 0].copy()

print(f"{len(pairs_all):,} pairs in total; {len(pairs):,} at different ages")
display(pairs.groupby("Comparison")["Bray_Curtis"].describe().round(3))

### 3.3 Overall: same infant vs. different infants (different ages only)
Descriptive overview. Because the mix of age gaps differs between the two groups, the age-matched
comparison in 3.4 is the more reliable one.

In [ ]:
order = ["Same infant", "Different infants"]

fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(
    data=pairs, x="Comparison", y="Bray_Curtis", order=order,
    hue="Comparison", palette=COMP_COLORS, legend=False,
    width=0.48, linewidth=1.5, showfliers=False,
    medianprops={"color": DARK, "linewidth": 2.5}, ax=ax,
)

medians = pairs.groupby("Comparison")["Bray_Curtis"].median()
for k, grp in enumerate(order):
    ax.text(k, medians[grp] + 0.045, f"Median: {medians[grp]:.2f}",
            ha="center", fontsize=12, fontweight="bold", color=DARK)

ax.set_title("Microbial community dissimilarity within vs. between infants", fontsize=16, fontweight="bold", pad=20)
ax.set_xlabel("")
ax.set_ylabel("Bray–Curtis dissimilarity")
ax.set_ylim(0, 1.05)
ax.grid(axis="x", visible=False)
sns.despine()
plt.tight_layout()
save_fig("bray_curtis_same_vs_different_infant.png")
plt.show()

### 3.4 Age-matched comparison by age gap
Only age gaps observed in *both* groups are kept, so same-infant and different-infant pairs are compared at equal time separation.

In [ ]:
gap_counts = pd.crosstab(pairs["Age_gap"], pairs["Comparison"])
valid_gaps = gap_counts.index[(gap_counts["Same infant"] > 0) & (gap_counts["Different infants"] > 0)]
pairs_matched = pairs[pairs["Age_gap"].isin(valid_gaps)].copy()

fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(
    data=pairs_matched, x="Age_gap", y="Bray_Curtis",
    hue="Comparison", hue_order=order, palette=COMP_COLORS,
    showfliers=False, width=0.7, ax=ax,
)
ax.set_title("Microbial dissimilarity by infant identity and age gap", fontsize=16, fontweight="bold")
ax.set_xlabel("Age difference between samples (months)")
ax.set_ylabel("Bray–Curtis dissimilarity")
ax.legend(title="", frameon=False, fontsize=11)
ax.grid(axis="x", visible=False)
sns.despine()
plt.tight_layout()
save_fig("bray_curtis_age_matched.png")
plt.show()

### 3.5 Exact age pairs: median dissimilarity, same vs. different infants
Same idea as 3.4 but at the finest resolution (e.g., 2→6 months). Δ = different-infant median − same-infant median;
positive Δ means samples from the same infant are more similar. The sample counts (`n_*`) show how much each estimate rests on.

In [ ]:
def age_pair_key(label):
    a, b = label.split("–")
    return (int(a), int(b))

summary = pairs.groupby(["Age_pair", "Comparison"])["Bray_Curtis"].agg(["median", "size"]).unstack()
summary.columns = [f"{'n' if stat == 'size' else 'median'}_{grp}" for stat, grp in summary.columns]
summary = summary.loc[sorted(summary.index, key=age_pair_key)]
summary["Difference"] = summary["median_Different infants"] - summary["median_Same infant"]
display(summary.round(3))

In [ ]:
age_labels = summary.index.tolist()
same_vals = summary["median_Same infant"].to_numpy()
diff_vals = summary["median_Different infants"].to_numpy()
delta = diff_vals - same_vals

x = np.arange(len(age_labels))
width = 0.34

fig, ax = plt.subplots(figsize=(14, 7))
bars1 = ax.bar(x - width/2, same_vals, width, label="Same infant", color="#63B3AD", edgecolor="white", linewidth=1.2)
bars2 = ax.bar(x + width/2, diff_vals, width, label="Different infants", color="#6272A4", edgecolor="white", linewidth=1.2)

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.012, f"{h:.2f}",
            ha="center", va="bottom", fontsize=10, color="#333333")

for k, d in enumerate(delta):
    ax.text(x[k], max(same_vals[k], diff_vals[k]) + 0.06, f"Δ={d:.2f}",
            ha="center", va="bottom", fontsize=10, fontweight="bold", color="#444444")

ax.set_xticks(x)
ax.set_xticklabels(age_labels, fontsize=12)
ax.set_xlabel("Age pair (months)")
ax.set_ylabel("Median Bray–Curtis dissimilarity")
ax.set_title("Median dissimilarity: same infant vs. different infants, by age pair", fontsize=18, fontweight="bold", pad=18)
ax.set_ylim(0, 1.08)
ax.legend(frameon=False, fontsize=12, loc="upper right")
ax.grid(axis="x", visible=False)
sns.despine()
plt.tight_layout()
save_fig("bray_curtis_exact_age_pairs.png")
plt.show()

### 3.6 Separating the two sources of variation
Two complementary views: how different are infants *from each other at the same age*, and how much does a *single infant* change across each age interval.

In [ ]:
between = pairs_all[(~pairs_all["Same_infant"]) & (pairs_all["Age_gap"] == 0)].copy()   # different infants, same age
within = pairs_all[pairs_all["Same_infant"] & (pairs_all["Age_gap"] > 0)].copy()        # same infant, different ages
within["Interval"] = within["Age_pair"]
interval_order = sorted(within["Interval"].unique(), key=age_pair_key)

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=between, x="Age1", y="Bray_Curtis", color=COMP_COLORS["Different infants"], showfliers=False, ax=ax)
ax.set_title("Between-infant microbial differences by age")
ax.set_xlabel("Infant age (months)")
ax.set_ylabel("Bray–Curtis dissimilarity")
sns.despine(); plt.tight_layout()
save_fig("between_infants_by_age.png")
plt.show()

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=within, x="Interval", y="Bray_Curtis", order=interval_order,
            color=COMP_COLORS["Same infant"], showfliers=False, ax=ax)
ax.set_title("Within-infant microbial changes across age intervals")
ax.set_xlabel("Age interval (months)")
ax.set_ylabel("Bray–Curtis dissimilarity")
ax.tick_params(axis="x", rotation=45)
sns.despine(); plt.tight_layout()
save_fig("within_infant_by_interval.png")
plt.show()

## 4. Variance partitioning: age vs. infant identity
PERMANOVA-style R² for one factor at a time: R² = 1 − SS_within / SS_total, computed directly from the distance matrix.

**Caution.** Raw R² grows with the number of groups. Infant ID has ~55 groups of ~4 samples each, whereas age has 5 groups,
so the raw R² for infant identity is inflated. The adjusted R² corrects for this and is the fair comparison.

In [ ]:
def variance_explained(D, groups):
    """R² = 1 - SS_within / SS_total for a distance matrix D and a grouping vector."""
    groups = np.asarray(groups)
    n = len(groups)
    ss_total = (D ** 2).sum() / (2 * n)
    ss_within = 0.0
    for g in np.unique(groups):
        idx = np.where(groups == g)[0]
        ss_within += (D[np.ix_(idx, idx)] ** 2).sum() / (2 * len(idx))
    return 1 - ss_within / ss_total

def adjusted_r2(r2, n, g):
    return 1 - (1 - r2) * (n - 1) / (n - g)

n = len(info)
rows = []
for label, col in [("Age", "age_months"), ("Infant (host_id)", "host_id")]:
    r2 = variance_explained(D, info[col])
    g = info[col].nunique()
    rows.append({"Factor": label, "Groups": g, "R²": r2, "Adjusted R²": adjusted_r2(r2, n, g)})

variance_table = pd.DataFrame(rows).set_index("Factor")
display(variance_table.round(3))

## 5. QIIME 2 PERMANOVA / PERMDISP (age)
Pre-computed with QIIME 2 (`beta-group-significance`). PERMANOVA tests whether group centroids differ;
PERMDISP tests whether group dispersions differ. A significant PERMANOVA is only cleanly interpretable as a
location effect if PERMDISP is not also significant.

In [ ]:
q2.Visualization.load(data_dir / "braycurtis_permanova_host_age.qzv")

In [ ]:
q2.Visualization.load(data_dir / "braycurtis_permdisp_host_age.qzv")